# Day 7: RAG, MCP and UiPath Integration for AI Automation  
## Insurance Domain Hands-on Lab Notebook

**Audience:** Beginner to Intermediate UiPath Developers, RPA Developers, Automation Engineers, Solution Architects  
**Duration:** 4 Hours  
**Domain:** Insurance Policy Knowledge, Claim Validation, Tool Access, MCP-style Integration, LangGraph, LangSmith and UiPath REST Integration

## Learning Outcomes

By the end of this notebook, participants will be able to:

- Understand RAG, embeddings and vector storage using insurance examples.
- Build a simple insurance policy RAG assistant.
- Use LangGraph with RAG for claim assistance.
- Understand MCP fundamentals: server, client, tools and tool discovery.
- Expose policy lookup and claim validation as AI tools.
- Understand LangSmith use cases for tracing LangGraph and MCP-style workflows.
- Invoke Python AI services from UiPath through REST API.
- Trigger a UiPath-supported automation step from an AI-assisted workflow.
- Build an AI Agent that invokes UiPath-supported insurance automation steps.

# 0. How to Use This Notebook

Run the notebook step by step.

Recommended flow:

1. Read concept explanation.
2. Run the setup cells.
3. Execute each insurance lab.
4. Modify claim/policy examples.
5. Complete mini assignments.
6. Review solution snippets at the end.

Environment variables are assumed to be configured:

```text
OPENROUTER_API_KEY=your_api_key_here
OPENROUTER_BASE_URL=https://openrouter.ai/api/v1
OPENROUTER_MODEL=openai/gpt-4.1-mini
```

Optional LangSmith variables:

```text
LANGSMITH_TRACING=true
LANGSMITH_API_KEY=your_langsmith_key_here
LANGSMITH_PROJECT=insurance-ai-automation-day7
```

# 1. Day 7 Agenda

| Time | Topic |
|---|---|
| 25 min | RAG, vector storage and embeddings |
| 35 min | Insurance policy RAG implementation |
| 30 min | LangGraph + RAG workflow |
| 35 min | MCP fundamentals and tool discovery |
| 30 min | Policy lookup and claim validation tools |
| 25 min | LangSmith use case |
| 35 min | UiPath REST integration pattern |
| 30 min | AI-assisted workflow triggering UiPath robot |
| 30 min | Final assignment and review |

Trainer humour:  
**Without RAG, the model may say “I think it is covered.” With RAG, it says “According to this policy clause…” Auditors prefer the second one.**

# 2. Install Required Libraries

Required packages:

- openai
- python-dotenv
- langchain
- langgraph
- fastapi
- uvicorn
- pydantic
- pandas
- numpy
- scikit-learn
- requests
- langsmith optional

In [ ]:
# Uncomment only if packages are missing.
# !pip install openai python-dotenv langchain langgraph fastapi uvicorn pydantic pandas numpy scikit-learn requests langsmith

# 3. API Setup and Reusable LLM Function

The same `call_llm()` function will be used for:

- RAG answer generation
- MCP-style tool responses
- LangGraph nodes
- UiPath service responses
- Final AI agent output

In [21]:
import os
import json
import time
from pathlib import Path
from typing import TypedDict, Dict, Any, List, Optional

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1")
OPENROUTER_MODEL = os.getenv("OPENROUTER_MODEL", "openai/gpt-4.1-mini")

print("Base URL:", OPENROUTER_BASE_URL)
print("Model:", OPENROUTER_MODEL)
print("API Key Loaded:", bool(OPENROUTER_API_KEY))

client = OpenAI(
    base_url=OPENROUTER_BASE_URL,
    api_key=OPENROUTER_API_KEY
)

def call_llm(prompt, system_message="You are an insurance AI automation assistant.", temperature=0.2):
    response = client.chat.completions.create(
        model=OPENROUTER_MODEL,
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": prompt}
        ],
        temperature=temperature
    )
    return response.choices[0].message.content

Base URL: https://openrouter.ai/api/v1
Model: openai/gpt-4.1-mini
API Key Loaded: True


# 4. Quick Introduction to RAG

RAG means **Retrieval-Augmented Generation**.

It combines:

1. Retrieval from documents or business knowledge  
2. LLM generation using retrieved context  

## Insurance Example

Question:

```text
Is room rent covered?
```

Retrieved policy clause:

```text
Room rent is covered up to INR 5,000 per day.
```

Final LLM answer:

```text
Yes. Room rent is covered up to INR 5,000 per day according to the policy clause.
```

# 5. Embeddings and Vector Storage

An embedding is a numerical representation of text.

Similar text has similar vectors.

Vector storage helps retrieve relevant policy clauses.

Examples:

- FAISS
- ChromaDB
- Pinecone
- Weaviate
- Azure AI Search
- pgvector

For beginner practice, we will use `TfidfVectorizer` from scikit-learn.

# 6. Sample Insurance Policy Knowledge Base

In [22]:
insurance_policy_documents = [
    {
        "doc_id": "POLICY_HEALTH_001",
        "title": "Health Secure Plus - Hospitalization Coverage",
        "text": "Health Secure Plus covers hospitalization expenses up to INR 5,00,000. Room rent is covered up to INR 5,000 per day."
    },
    {
        "doc_id": "POLICY_HEALTH_002",
        "title": "Health Secure Plus - Pre and Post Hospitalization",
        "text": "Pre-hospitalization expenses are covered for 30 days. Post-hospitalization expenses are covered for 60 days."
    },
    {
        "doc_id": "POLICY_HEALTH_003",
        "title": "Health Secure Plus - Exclusions",
        "text": "Cosmetic treatment, non-prescribed medicines and experimental procedures are excluded from coverage."
    },
    {
        "doc_id": "POLICY_MOTOR_001",
        "title": "Comprehensive Motor Policy - Own Damage",
        "text": "Comprehensive motor insurance covers own damage due to accident, theft, fire and natural calamities."
    },
    {
        "doc_id": "POLICY_MOTOR_002",
        "title": "Motor Claim Documents",
        "text": "Motor claim documents include repair estimate, vehicle photos, police report, driving license and RC copy."
    },
    {
        "doc_id": "POLICY_TRAVEL_001",
        "title": "Travel Insurance - Baggage Delay",
        "text": "Baggage delay is covered if baggage is delayed by more than 12 hours. Required documents include boarding pass and airline delay certificate."
    },
    {
        "doc_id": "POLICY_TRAVEL_002",
        "title": "Travel Insurance - Exclusions",
        "text": "Travel insurance excludes losses due to unattended baggage, invalid travel documents and intentional delay."
    }
]

print("Insurance knowledge base loaded:", len(insurance_policy_documents), "documents")

Insurance knowledge base loaded: 7 documents


# 7. Lab 1: Build Simple Vector Retrieval

This lab uses TF-IDF to simulate vector retrieval.

Flow:

```text
Question → Vectorize → Compare with policy clauses → Return top matches
```

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def rebuild_vector_store():
    global vectorizer, document_vectors
    documents_text = [doc["text"] for doc in insurance_policy_documents]
    vectorizer = TfidfVectorizer()
    document_vectors = vectorizer.fit_transform(documents_text)

def retrieve_relevant_documents(query, top_k=3):
    query_vector = vectorizer.transform([query])
    similarities = cosine_similarity(query_vector, document_vectors).flatten()
    ranked_indices = similarities.argsort()[::-1][:top_k]

    results = []
    for index in ranked_indices:
        results.append({
            "score": float(similarities[index]),
            "doc_id": insurance_policy_documents[index]["doc_id"],
            "title": insurance_policy_documents[index]["title"],
            "text": insurance_policy_documents[index]["text"]
        })
    return results

rebuild_vector_store()

query = "Is room rent covered in health policy?"
retrieved_docs = retrieve_relevant_documents(query)

for doc in retrieved_docs:
    print(doc["score"], doc["title"])
    print(doc["text"])
    print("-" * 80)

0.40729022574825596 Health Secure Plus - Hospitalization Coverage
Health Secure Plus covers hospitalization expenses up to INR 5,00,000. Room rent is covered up to INR 5,000 per day.
--------------------------------------------------------------------------------
0.19478404807843877 Travel Insurance - Baggage Delay
Baggage delay is covered if baggage is delayed by more than 12 hours. Required documents include boarding pass and airline delay certificate.
--------------------------------------------------------------------------------
0.10417336622448785 Health Secure Plus - Pre and Post Hospitalization
Pre-hospitalization expenses are covered for 30 days. Post-hospitalization expenses are covered for 60 days.
--------------------------------------------------------------------------------


## Mini Assignment 1

Try these questions:

1. What documents are required for motor claim?
2. Is cosmetic treatment covered?
3. Is baggage delay covered?
4. What are travel insurance exclusions?

In [4]:
practice_questions = [
    "What documents are required for motor claim?",
    "Is cosmetic treatment covered?",
    "Is baggage delay covered?",
    "What are travel insurance exclusions?"
]

for question in practice_questions:
    print("Question:", question)
    docs = retrieve_relevant_documents(question, top_k=2)
    for doc in docs:
        print("-", doc["title"], "| Score:", round(doc["score"], 3))
    print()

Question: What documents are required for motor claim?
- Health Secure Plus - Pre and Post Hospitalization | Score: 0.324
- Motor Claim Documents | Score: 0.273

Question: Is cosmetic treatment covered?
- Health Secure Plus - Exclusions | Score: 0.338
- Travel Insurance - Baggage Delay | Score: 0.223

Question: Is baggage delay covered?
- Travel Insurance - Baggage Delay | Score: 0.613
- Travel Insurance - Exclusions | Score: 0.238

Question: What are travel insurance exclusions?
- Travel Insurance - Exclusions | Score: 0.483
- Health Secure Plus - Pre and Post Hospitalization | Score: 0.189



# 8. Lab 2: RAG Answer Generation

Now we combine retrieval with LLM generation.

Flow:

```text
Question → Retrieve context → Prompt LLM → Grounded answer
```

In [23]:
def rag_answer(question, top_k=3):
    docs = retrieve_relevant_documents(question, top_k=top_k)
    context = "\n\n".join([
        "Title: " + doc["title"] + "\nText: " + doc["text"]
        for doc in docs
    ])

    prompt = (
        "You are an insurance policy assistant.\n"
        "Answer the user question using only the provided context.\n"
        "Rules:\n"
        "- Do not invent policy details.\n"
        "- If answer is not available, say information is not available.\n"
        "- Mention source titles briefly.\n\n"
        "Context:\n" + context + "\n\n"
        "Question:\n" + question
    )

    answer = call_llm(prompt)
    return {
        "question": question,
        "retrieved_documents": docs,
        "answer": answer
    }

# Uncomment after API key setup.
result = rag_answer("Is room rent covered in Health Secure Plus?")
print(result["answer"])

Yes, room rent is covered in Health Secure Plus up to INR 5,000 per day. (Source: Health Secure Plus - Hospitalization Coverage)


## Mini Assignment 2

Create RAG answers for:

1. Motor claim documents  
2. Baggage delay coverage  
3. Health policy exclusions  
4. Pre-hospitalization coverage

In [7]:
# Uncomment after API key setup.
for question in practice_questions:
    result = rag_answer(question, top_k=2)
    print("Question:", result["question"])
    print("Answer:", result["answer"])
    print("Sources:", [doc["title"] for doc in result["retrieved_documents"]])
    print("-" * 100)

Question: What documents are required for motor claim?
Answer: The documents required for a motor claim are repair estimate, vehicle photos, police report, driving license, and RC copy. (Source: Motor Claim Documents)
Sources: ['Health Secure Plus - Pre and Post Hospitalization', 'Motor Claim Documents']
----------------------------------------------------------------------------------------------------
Question: Is cosmetic treatment covered?
Answer: Cosmetic treatment is excluded from coverage according to the Health Secure Plus - Exclusions.
Sources: ['Health Secure Plus - Exclusions', 'Travel Insurance - Baggage Delay']
----------------------------------------------------------------------------------------------------
Question: Is baggage delay covered?
Answer: Yes, baggage delay is covered if the baggage is delayed by more than 12 hours. Required documents include the boarding pass and airline delay certificate. (Source: Travel Insurance - Baggage Delay)
Sources: ['Travel Insuran

# 9. Lab 3: RAG with Structured JSON Output for UiPath

UiPath prefers structured JSON.

We will ask the RAG assistant to return:

- answer
- confidence
- source_titles
- recommended_action

In [9]:
def rag_answer_json(question, top_k=3):
    docs = retrieve_relevant_documents(question, top_k=top_k)
    context = "\n\n".join([
        "Title: " + doc["title"] + "\nText: " + doc["text"]
        for doc in docs
    ])

    prompt = (
        "You are an insurance policy assistant.\n"
        "Use only the context to answer. Return valid JSON only.\n"
        "Schema:\n"
        "{\n"
        "  \"answer\": \"\",\n"
        "  \"confidence\": \"High/Medium/Low\",\n"
        "  \"source_titles\": [],\n"
        "  \"recommended_action\": \"\"\n"
        "}\n\n"
        "Context:\n" + context + "\n\n"
        "Question:\n" + question
    )

    response = call_llm(prompt)

    try:
        return json.loads(response)
    except Exception:
        return {
            "answer": response,
            "confidence": "Medium",
            "source_titles": [doc["title"] for doc in docs],
            "recommended_action": "Manual JSON review required"
        }

# Uncomment after API key setup.
json_result = rag_answer_json("What documents are required for motor claim?")
print(json.dumps(json_result, indent=2))

{
  "answer": "The documents required for a motor claim include repair estimate, vehicle photos, police report, driving license, and RC copy.",
  "confidence": "High",
  "source_titles": [
    "Motor Claim Documents"
  ],
  "recommended_action": "Collect and submit the repair estimate, vehicle photos, police report, driving license, and RC copy to process the motor claim."
}


## Mini Assignment 3

Enhance JSON output to include:

- policy_area
- claim_type
- requires_human_review

# 10. LangGraph + RAG Workflow

Use case:

```text
Customer Question
   ↓
Retrieve policy context
   ↓
Generate answer
   ↓
Check confidence
   ↓
Return final response or human review message
```

In [10]:
from langgraph.graph import StateGraph, START, END

class RAGWorkflowState(TypedDict, total=False):
    question: str
    retrieved_documents: List[Dict[str, Any]]
    answer: str
    confidence: str
    human_review_required: bool
    final_response: str
    logs: List[str]

def retrieval_node(state: RAGWorkflowState) -> RAGWorkflowState:
    docs = retrieve_relevant_documents(state["question"], top_k=3)
    logs = state.get("logs", [])
    logs.append("Retrieval node completed.")
    return {**state, "retrieved_documents": docs, "logs": logs}

def answer_generation_node(state: RAGWorkflowState) -> RAGWorkflowState:
    docs = state["retrieved_documents"]
    context = "\n\n".join([doc["title"] + ": " + doc["text"] for doc in docs])
    prompt = (
        "You are an insurance RAG assistant.\n"
        "Answer using only the context.\n"
        "Return answer and confidence.\n\n"
        "Context:\n" + context + "\n\n"
        "Question:\n" + state["question"]
    )
    response = call_llm(prompt)
    confidence = "Low" if "not available" in response.lower() else "High"
    logs = state.get("logs", [])
    logs.append("Answer generation node completed.")
    return {**state, "answer": response, "confidence": confidence, "logs": logs}

def rag_review_node(state: RAGWorkflowState) -> RAGWorkflowState:
    review = state.get("confidence") == "Low"
    logs = state.get("logs", [])
    logs.append("RAG review node completed.")
    return {**state, "human_review_required": review, "logs": logs}

def final_rag_response_node(state: RAGWorkflowState) -> RAGWorkflowState:
    if state.get("human_review_required"):
        final_response = "Human review required before customer response. Draft answer: " + state.get("answer", "")
    else:
        final_response = state.get("answer", "")
    logs = state.get("logs", [])
    logs.append("Final RAG response node completed.")
    return {**state, "final_response": final_response, "logs": logs}

rag_graph = StateGraph(RAGWorkflowState)
rag_graph.add_node("retrieve", retrieval_node)
rag_graph.add_node("generate_answer", answer_generation_node)
rag_graph.add_node("review", rag_review_node)
rag_graph.add_node("final", final_rag_response_node)

rag_graph.add_edge(START, "retrieve")
rag_graph.add_edge("retrieve", "generate_answer")
rag_graph.add_edge("generate_answer", "review")
rag_graph.add_edge("review", "final")
rag_graph.add_edge("final", END)

rag_app = rag_graph.compile()
print("LangGraph RAG workflow compiled.")

LangGraph RAG workflow compiled.


In [12]:
# Uncomment after API key setup.
result = rag_app.invoke({
    "question": "Is cosmetic treatment covered in Health Secure Plus?",
    "logs": []
})
print(result["final_response"])
print(result["logs"])


Answer: No, cosmetic treatment is excluded from coverage in Health Secure Plus.

Confidence: High
['Retrieval node completed.', 'Answer generation node completed.', 'RAG review node completed.', 'Final RAG response node completed.']


## Mini Assignment 4

Add conditional routing:

- Low confidence → Human Review Node
- High/Medium confidence → Final Response Node

# 11. Retry and Error Recovery

RAG and tool calls may fail.

Common causes:

- API failure
- empty context
- invalid JSON
- timeout
- network error

In [13]:
def call_llm_with_retry(prompt, max_retries=3, delay_seconds=2):
    errors = []
    for attempt in range(1, max_retries + 1):
        try:
            result = call_llm(prompt)
            return result, errors
        except Exception as error:
            errors.append(f"Attempt {attempt} failed: {str(error)}")
            time.sleep(delay_seconds * attempt)
    return None, errors

def safe_rag_answer(question):
    docs = retrieve_relevant_documents(question, top_k=3)
    context = "\n\n".join([doc["text"] for doc in docs])
    prompt = "Answer this insurance question using context only.\nContext:\n" + context + "\nQuestion:\n" + question
    result, errors = call_llm_with_retry(prompt)
    if result is None:
        return {
            "status": "failed",
            "answer": "AI service unavailable. Route to manual review.",
            "errors": errors
        }
    return {
        "status": "success",
        "answer": result,
        "errors": errors
    }

# Uncomment after API key setup.
print(safe_rag_answer("What is covered under motor policy?"))

{'status': 'success', 'answer': 'The provided context does not include any information about what is covered under a motor policy.', 'errors': []}


# 12. MCP Fundamentals

MCP stands for **Model Context Protocol**.

In enterprise AI automation, MCP-style design helps AI systems discover and use tools safely.

## MCP Concepts

| Concept | Meaning |
|---|---|
| MCP Server | Exposes tools or resources |
| MCP Client | Connects to server and uses tools |
| Tool | Callable business function |
| Tool Discovery | AI/client can list available tools |
| Governance | Controls who can access tools |

Insurance tool examples:

- policy_lookup_tool
- claim_validation_tool
- document_checklist_tool
- uipath_robot_trigger_tool

# 13. Lab 4: Create MCP-Style Tools

We will create:

1. Policy Lookup Tool  
2. Claim Validation Tool  
3. Document Checklist Tool

In [14]:
policy_database = {
    "POL1001": {
        "customer_name": "Rajesh Sharma",
        "policy_type": "Health",
        "policy_status": "Active",
        "premium_status": "Paid",
        "coverage_amount": 500000
    },
    "VEH2045": {
        "customer_name": "Meena Joshi",
        "policy_type": "Motor",
        "policy_status": "Active",
        "premium_status": "Paid",
        "coverage_amount": 300000
    },
    "TRV3001": {
        "customer_name": "Amit Verma",
        "policy_type": "Travel",
        "policy_status": "Active",
        "premium_status": "Paid",
        "coverage_amount": 100000
    }
}

required_documents_by_type = {
    "Health": ["Hospital Bill", "Discharge Summary", "Doctor Prescription", "Claim Form"],
    "Motor": ["Repair Estimate", "Vehicle Photos", "Police Report", "Driving License", "RC Copy"],
    "Travel": ["Boarding Pass", "Airline Delay Certificate", "Baggage Delay Report", "Claim Form"],
    "Property": ["Damage Photos", "Repair Estimate", "Ownership Proof", "Claim Form"]
}

def policy_lookup_tool(policy_number):
    return policy_database.get(policy_number, {"error": "Policy not found"})

def document_checklist_tool(claim_type):
    return required_documents_by_type.get(claim_type, [])

def claim_validation_tool(claim):
    policy = policy_lookup_tool(claim.get("policy_number"))
    if "error" in policy:
        return {"valid": False, "reason": "Policy not found", "next_action": "Route to policy exception queue"}
    if policy["policy_status"] != "Active":
        return {"valid": False, "reason": "Policy inactive", "next_action": "Route to policy exception queue"}
    if policy["premium_status"] != "Paid":
        return {"valid": False, "reason": "Premium pending", "next_action": "Request premium payment verification"}
    if claim.get("claim_amount", 0) > policy["coverage_amount"]:
        return {"valid": False, "reason": "Claim exceeds coverage", "next_action": "Route to human review"}
    return {"valid": True, "reason": "Claim is within active policy coverage", "next_action": "Proceed to document and risk review"}

sample_claim = {
    "claim_id": "CLM9001",
    "policy_number": "POL1001",
    "customer_name": "Rajesh Sharma",
    "claim_type": "Health",
    "claim_amount": 85000
}

print(policy_lookup_tool("POL1001"))
print(document_checklist_tool("Health"))
print(claim_validation_tool(sample_claim))

{'customer_name': 'Rajesh Sharma', 'policy_type': 'Health', 'policy_status': 'Active', 'premium_status': 'Paid', 'coverage_amount': 500000}
['Hospital Bill', 'Discharge Summary', 'Doctor Prescription', 'Claim Form']
{'valid': True, 'reason': 'Claim is within active policy coverage', 'next_action': 'Proceed to document and risk review'}


# 14. Lab 5: Tool Discovery

Tool discovery allows an AI client to know which tools are available.

In [15]:
tool_registry = {
    "policy_lookup_tool": {
        "description": "Looks up insurance policy details by policy number.",
        "input_schema": {"policy_number": "str"},
        "output": "policy details"
    },
    "claim_validation_tool": {
        "description": "Validates claim amount and policy status.",
        "input_schema": {"claim": "dict"},
        "output": "validation result"
    },
    "document_checklist_tool": {
        "description": "Returns required documents by claim type.",
        "input_schema": {"claim_type": "str"},
        "output": "document checklist"
    }
}

def discover_tools():
    return tool_registry

def run_tool(tool_name, tool_input):
    if tool_name == "policy_lookup_tool":
        return policy_lookup_tool(tool_input["policy_number"])
    if tool_name == "claim_validation_tool":
        return claim_validation_tool(tool_input["claim"])
    if tool_name == "document_checklist_tool":
        return document_checklist_tool(tool_input["claim_type"])
    return {"error": "Tool not found"}

print(json.dumps(discover_tools(), indent=2))
print(run_tool("document_checklist_tool", {"claim_type": "Motor"}))

{
  "policy_lookup_tool": {
    "description": "Looks up insurance policy details by policy number.",
    "input_schema": {
      "policy_number": "str"
    },
    "output": "policy details"
  },
  "claim_validation_tool": {
    "description": "Validates claim amount and policy status.",
    "input_schema": {
      "claim": "dict"
    },
    "output": "validation result"
  },
  "document_checklist_tool": {
    "description": "Returns required documents by claim type.",
    "input_schema": {
      "claim_type": "str"
    },
    "output": "document checklist"
  }
}
['Repair Estimate', 'Vehicle Photos', 'Police Report', 'Driving License', 'RC Copy']


# 15. Lab 6: AI Agent Choosing a Tool

The AI agent receives a question and decides which tool to call.

In [16]:
def detect_tool_from_question(question):
    q = question.lower()
    if "document" in q or "checklist" in q or "submit" in q:
        return "document_checklist_tool"
    if "policy" in q or "coverage" in q:
        return "policy_lookup_tool"
    if "validate" in q or "claim amount" in q or "eligible" in q:
        return "claim_validation_tool"
    return None

def ai_tool_agent(question, context):
    tool_name = detect_tool_from_question(question)

    if tool_name == "policy_lookup_tool":
        tool_result = run_tool("policy_lookup_tool", {"policy_number": context.get("policy_number")})
    elif tool_name == "claim_validation_tool":
        tool_result = run_tool("claim_validation_tool", {"claim": context})
    elif tool_name == "document_checklist_tool":
        tool_result = run_tool("document_checklist_tool", {"claim_type": context.get("claim_type")})
    else:
        tool_result = {"message": "No tool selected"}

    prompt = (
        "You are an insurance AI tool agent.\n"
        "User Question:\n" + question + "\n\n"
        "Selected Tool:\n" + str(tool_name) + "\n\n"
        "Tool Result:\n" + json.dumps(tool_result, indent=2) + "\n\n"
        "Create a professional response for an automation user."
    )

    response = call_llm(prompt)
    return {"tool_name": tool_name, "tool_result": tool_result, "response": response}

# Uncomment after API key setup.
result = ai_tool_agent("What documents should I submit for this claim?", sample_claim)
print(result["response"])

To process your claim smoothly, please submit the following documents:

- Hospital Bill  
- Discharge Summary  
- Doctor Prescription  
- Completed Claim Form  

Ensure all documents are clear and legible to avoid any delays. If you need further assistance, feel free to ask.


# 16. Simple MCP-Style Tool Server using FastAPI

This creates a FastAPI app exposing:

- `/tools`
- `/tool/policy-lookup`
- `/tool/claim-validation`
- `/tool/document-checklist`

In [ ]:
mcp_server_lines = [
'from fastapi import FastAPI',
'from pydantic import BaseModel',
'from typing import Dict, Any',
'',
'app = FastAPI(title="Insurance MCP-Style Tool Server")',
'',
'policy_database = {',
'    "POL1001": {"customer_name": "Rajesh Sharma", "policy_type": "Health", "policy_status": "Active", "premium_status": "Paid", "coverage_amount": 500000},',
'    "VEH2045": {"customer_name": "Meena Joshi", "policy_type": "Motor", "policy_status": "Active", "premium_status": "Paid", "coverage_amount": 300000}',
'}',
'',
'required_documents = {',
'    "Health": ["Hospital Bill", "Discharge Summary", "Doctor Prescription", "Claim Form"],',
'    "Motor": ["Repair Estimate", "Vehicle Photos", "Police Report", "Driving License", "RC Copy"]',
'}',
'',
'class PolicyLookupRequest(BaseModel):',
'    policy_number: str',
'',
'class ClaimValidationRequest(BaseModel):',
'    claim: Dict[str, Any]',
'',
'class DocumentChecklistRequest(BaseModel):',
'    claim_type: str',
'',
'@app.get("/tools")',
'def list_tools():',
'    return {"tools": ["policy_lookup_tool", "claim_validation_tool", "document_checklist_tool"]}',
'',
'@app.post("/tool/policy-lookup")',
'def policy_lookup(request: PolicyLookupRequest):',
'    return policy_database.get(request.policy_number, {"error": "Policy not found"})',
'',
'@app.post("/tool/claim-validation")',
'def claim_validation(request: ClaimValidationRequest):',
'    claim = request.claim',
'    policy = policy_database.get(claim.get("policy_number"))',
'    if not policy:',
'        return {"valid": False, "reason": "Policy not found"}',
'    if claim.get("claim_amount", 0) > policy["coverage_amount"]:',
'        return {"valid": False, "reason": "Claim exceeds coverage"}',
'    return {"valid": True, "reason": "Claim is within coverage"}',
'',
'@app.post("/tool/document-checklist")',
'def document_checklist(request: DocumentChecklistRequest):',
'    return {"claim_type": request.claim_type, "required_documents": required_documents.get(request.claim_type, [])}',
]

Path("mcp_tool_server.py").write_text("\n".join(mcp_server_lines), encoding="utf-8")
print("mcp_tool_server.py created.")
print("Run: uvicorn mcp_tool_server:app --reload --port 8001")
print("Open: http://127.0.0.1:8001/docs")

# 17. Quick Introduction to LangSmith

LangSmith is used for observability, debugging, tracing and evaluation of LLM applications.

Use cases:

- Trace LangChain chains
- Debug LangGraph node execution
- Track inputs and outputs
- Monitor tool calls
- Evaluate RAG quality
- Support audit and governance discussions

In [ ]:
os.environ["LANGSMITH_TRACING"] = os.getenv("LANGSMITH_TRACING", "false")
os.environ["LANGSMITH_PROJECT"] = os.getenv("LANGSMITH_PROJECT", "insurance-day7-rag-mcp")

print("LangSmith tracing:", os.environ.get("LANGSMITH_TRACING"))
print("LangSmith project:", os.environ.get("LANGSMITH_PROJECT"))

# 18. LangSmith Use Case with LangGraph and MCP

Scenario:

```text
Customer asks: Is my claim eligible under policy POL1001?
```

Workflow:

1. LangGraph receives question.
2. MCP-style tool performs policy lookup.
3. Claim validation tool checks eligibility.
4. LLM generates response.
5. LangSmith traces the steps when enabled.

This helps answer:

- Which tool was called?
- What input was sent?
- What output was returned?
- Where did workflow fail?
- Was human review needed?

# 19. UiPath Integration Patterns with Python and REST APIs

UiPath can integrate with Python AI services using:

| Pattern | Description |
|---|---|
| HTTP Request Activity | UiPath calls FastAPI endpoint |
| Invoke Python | UiPath calls local Python script |
| Queue-based Integration | UiPath queue triggers AI backend |
| Orchestrator API | AI backend triggers UiPath job |
| File-based Integration | UiPath and Python exchange JSON/CSV |

Recommended pattern:

```text
UiPath Robot → HTTP Request → Python FastAPI AI Service → LLM/RAG/MCP Tools → JSON Response
```

# 20. Lab 7: Create Python AI Service for UiPath

Endpoint:

```text
POST /ai-policy-assistant
```

This endpoint returns JSON suitable for UiPath workflow decisions.

In [ ]:
uipath_service_lines = [
'import os',
'from fastapi import FastAPI',
'from pydantic import BaseModel',
'from typing import Optional',
'from dotenv import load_dotenv',
'from openai import OpenAI',
'',
'load_dotenv()',
'OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")',
'OPENROUTER_BASE_URL = os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1")',
'OPENROUTER_MODEL = os.getenv("OPENROUTER_MODEL", "openai/gpt-4.1-mini")',
'client = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=OPENROUTER_API_KEY)',
'',
'app = FastAPI(title="Insurance AI Service for UiPath")',
'',
'policy_database = {',
'    "POL1001": {"customer_name": "Rajesh Sharma", "policy_type": "Health", "policy_status": "Active", "premium_status": "Paid", "coverage_amount": 500000}',
'}',
'',
'class PolicyAssistantRequest(BaseModel):',
'    question: str',
'    policy_number: str',
'    claim_amount: Optional[float] = 0',
'',
'def call_llm(prompt):',
'    response = client.chat.completions.create(',
'        model=OPENROUTER_MODEL,',
'        messages=[{"role": "system", "content": "You are an insurance AI automation assistant."}, {"role": "user", "content": prompt}],',
'        temperature=0.2',
'    )',
'    return response.choices[0].message.content',
'',
'@app.post("/ai-policy-assistant")',
'def ai_policy_assistant(request: PolicyAssistantRequest):',
'    policy = policy_database.get(request.policy_number)',
'    if not policy:',
'        return {"status": "failed", "reason": "Policy not found", "next_action": "Route to policy exception queue"}',
'    prompt = "Question: " + request.question + "\\nPolicy: " + str(policy) + "\\nClaim Amount: " + str(request.claim_amount)',
'    answer = call_llm(prompt)',
'    return {"status": "success", "policy_number": request.policy_number, "answer": answer, "policy_status": policy["policy_status"], "coverage_amount": policy["coverage_amount"]}',
]

Path("uipath_ai_service.py").write_text("\n".join(uipath_service_lines), encoding="utf-8")
print("uipath_ai_service.py created.")
print("Run: uvicorn uipath_ai_service:app --reload --port 8002")
print("Open: http://127.0.0.1:8002/docs")

## UiPath Testing Steps

1. Start FastAPI:

```bash
uvicorn uipath_ai_service:app --reload --port 8002
```

2. Open Swagger UI:

```text
http://127.0.0.1:8002/docs
```

3. In UiPath, use **HTTP Request** activity.

4. Configure:

```text
Method: POST
Endpoint: http://127.0.0.1:8002/ai-policy-assistant
Body Type: JSON
```

5. Sample JSON:

```json
{
  "question": "Is this claim eligible under the policy?",
  "policy_number": "POL1001",
  "claim_amount": 85000
}
```

# 21. Trigger UiPath Robot from AI-Assisted Workflow

For training, we simulate UiPath Orchestrator trigger using a mock function.

In [17]:
def trigger_uipath_robot(process_name, payload):
    return {
        "trigger_status": "success",
        "process_name": process_name,
        "job_id": "JOB-MOCK-1001",
        "payload": payload,
        "message": "UiPath robot trigger simulated successfully"
    }

def ai_assisted_uipath_workflow(claim):
    validation = claim_validation_tool(claim)

    if validation["valid"]:
        process_name = "Claim_Auto_Processing_Robot"
    elif validation["reason"] == "Claim exceeds coverage":
        process_name = "Human_Review_Robot"
    elif validation["reason"] == "Policy not found":
        process_name = "Policy_Exception_Robot"
    else:
        process_name = "Claim_Exception_Handling_Robot"

    trigger_result = trigger_uipath_robot(process_name, {
        "claim": claim,
        "validation": validation
    })

    return {
        "validation": validation,
        "uipath_trigger": trigger_result
    }

result = ai_assisted_uipath_workflow(sample_claim)
print(json.dumps(result, indent=2))

{
  "validation": {
    "valid": true,
    "reason": "Claim is within active policy coverage",
    "next_action": "Proceed to document and risk review"
  },
  "uipath_trigger": {
    "trigger_status": "success",
    "process_name": "Claim_Auto_Processing_Robot",
    "job_id": "JOB-MOCK-1001",
    "payload": {
      "claim": {
        "claim_id": "CLM9001",
        "policy_number": "POL1001",
        "customer_name": "Rajesh Sharma",
        "claim_type": "Health",
        "claim_amount": 85000
      },
      "validation": {
        "valid": true,
        "reason": "Claim is within active policy coverage",
        "next_action": "Proceed to document and risk review"
      }
    },
    "message": "UiPath robot trigger simulated successfully"
  }
}


# 22. Security and Governance Considerations

AI tool access must be governed.

| Area | Governance Control |
|---|---|
| API Keys | Store in environment variables or vault |
| Tool Access | Allow only approved tools |
| Input Validation | Validate policy number, claim amount, claim type |
| Output Validation | Return structured JSON |
| Audit Trail | Log tool used, input, output, timestamp |
| Human Review | Required for low confidence or high-risk cases |
| Data Privacy | Minimize PII |
| UiPath Trigger | Trigger only approved processes |

Rule: **AI recommends. Workflow decides. Humans approve exceptions.**

# 23. Example 1: Health Policy RAG Assistant

Ask:

```text
Are pre-hospitalization expenses covered?
```

In [18]:
# Uncomment after API key setup.
# result = rag_answer("Are pre-hospitalization expenses covered in Health Secure Plus?")
# print(result["answer"])
# print("Sources:", [doc["title"] for doc in result["retrieved_documents"]])

In [19]:
# Uncommented selection: run a RAG example (requires API key / client already configured)
result = rag_answer("Are pre-hospitalization expenses covered in Health Secure Plus?")
print(result["answer"])
print("Sources:", [doc["title"] for doc in result["retrieved_documents"]])

Yes, pre-hospitalization expenses are covered for 30 days in Health Secure Plus. (Source: Health Secure Plus - Pre and Post Hospitalization)
Sources: ['Health Secure Plus - Pre and Post Hospitalization', 'Health Secure Plus - Hospitalization Coverage', 'Health Secure Plus - Exclusions']


## Assignment Example 1

Add a new policy clause:

```text
Daycare procedures are covered when medically necessary.
```

Then ask:

```text
Are daycare procedures covered?
``` 

In [20]:
daycare_clause = {
    "doc_id": "POLICY_HEALTH_004",
    "title": "Health Secure Plus - Daycare Procedures",
    "text": "Daycare procedures are covered when medically necessary."
}

insurance_policy_documents.append(daycare_clause)
rebuild_vector_store()

question = "Are daycare procedures covered?"
result = rag_answer(question, top_k=3)

print(result["answer"])
print("Sources:", [doc["title"] for doc in result["retrieved_documents"]])

Yes, daycare procedures are covered when medically necessary. (Source: Health Secure Plus - Daycare Procedures)
Sources: ['Health Secure Plus - Daycare Procedures', 'Health Secure Plus - Pre and Post Hospitalization', 'Health Secure Plus - Exclusions']


# 24. Example 2: Motor Claim Tool Agent

In [ ]:
motor_claim = {
    "claim_id": "MOT2001",
    "policy_number": "VEH2045",
    "customer_name": "Meena Joshi",
    "claim_type": "Motor",
    "claim_amount": 240000
}

# Uncomment after API key setup.
# result = ai_tool_agent("Is this motor claim eligible under the policy?", motor_claim)
# print(result["response"])
# print(result["tool_result"])

## Assignment Example 2

Increase motor claim amount to INR 3,50,000 and observe validation result.

# 25. Example 3: Travel Claim RAG + UiPath Trigger

In [ ]:
travel_claim = {
    "claim_id": "TRV3001",
    "policy_number": "TRV3001",
    "customer_name": "Amit Verma",
    "claim_type": "Travel",
    "claim_amount": 18000
}

travel_result = ai_assisted_uipath_workflow(travel_claim)
print(json.dumps(travel_result, indent=2))

## Assignment Example 3

Add logic:

If claim type is Travel and amount is less than INR 25,000, trigger:

```text
Travel_Express_Claim_Robot
```

# 26. Example 4: MCP Tool Discovery for AI Agent

In [ ]:
def suggest_tool_with_llm(user_request):
    tools = discover_tools()
    prompt = (
        "You are an insurance AI tool selection assistant.\n\n"
        "Available tools:\n" + json.dumps(tools, indent=2) + "\n\n"
        "User request:\n" + user_request + "\n\n"
        "Return selected_tool, reason and required_input."
    )
    return call_llm(prompt)

# Uncomment after API key setup.
# print(suggest_tool_with_llm("Which documents are required for motor claim?"))

# 27. Example 5: LangGraph RAG with Human Review

In [ ]:
class RAGHumanReviewState(TypedDict, total=False):
    question: str
    answer: str
    confidence: str
    human_review_required: bool
    final_message: str
    logs: List[str]

def simple_rag_answer_node(state: RAGHumanReviewState) -> RAGHumanReviewState:
    result = rag_answer_json(state["question"])
    logs = state.get("logs", [])
    logs.append("RAG answer JSON node completed.")
    return {
        **state,
        "answer": result.get("answer", ""),
        "confidence": result.get("confidence", "Medium"),
        "logs": logs
    }

def decide_human_review_node(state: RAGHumanReviewState) -> RAGHumanReviewState:
    review = state.get("confidence", "Medium") == "Low"
    logs = state.get("logs", [])
    logs.append("Human review decision node completed.")
    return {**state, "human_review_required": review, "logs": logs}

def rag_human_final_node(state: RAGHumanReviewState) -> RAGHumanReviewState:
    if state.get("human_review_required"):
        message = "Human review required before sending answer. Draft answer: " + state.get("answer", "")
    else:
        message = state.get("answer", "")
    logs = state.get("logs", [])
    logs.append("Final response node completed.")
    return {**state, "final_message": message, "logs": logs}

print("Human review RAG nodes created.")

# 28. Final Assignment: Build AI Agent that Invokes UiPath-Supported Insurance Automation Steps

## Assignment Objective

Build an AI Agent that can:

1. Answer policy questions using RAG.
2. Discover available MCP-style tools.
3. Use policy lookup tool.
4. Use claim validation tool.
5. Decide next automation route.
6. Trigger mock UiPath robot.
7. Return structured JSON for UiPath or audit.

## Required Workflow

```text
User / Claim Input
   ↓
RAG Policy Question Answering
   ↓
MCP Tool Discovery
   ↓
Policy Lookup / Claim Validation Tool
   ↓
LangGraph Routing
   ↓
UiPath Robot Trigger
   ↓
Customer Response + Audit Output
```

# 29. Assignment Solutions

## Solution 1: Add Daycare Procedure Clause

In [ ]:
daycare_clause = {
    "doc_id": "POLICY_HEALTH_004",
    "title": "Health Secure Plus - Daycare Procedures",
    "text": "Daycare procedures are covered when medically necessary and approved as per policy terms."
}

insurance_policy_documents.append(daycare_clause)
rebuild_vector_store()

print("Daycare clause added and vector store rebuilt.")

## Solution 2: Enhanced Travel UiPath Trigger

In [ ]:
def travel_aware_uipath_workflow(claim):
    validation = claim_validation_tool(claim)

    if claim.get("claim_type") == "Travel" and claim.get("claim_amount", 0) < 25000 and validation["valid"]:
        process_name = "Travel_Express_Claim_Robot"
    elif validation["valid"]:
        process_name = "Claim_Auto_Processing_Robot"
    else:
        process_name = "Claim_Exception_Handling_Robot"

    return trigger_uipath_robot(process_name, {
        "claim": claim,
        "validation": validation
    })

print(json.dumps(travel_aware_uipath_workflow(travel_claim), indent=2))

## Solution 3: Simple Governance Audit Record

In [ ]:
def create_audit_record(user_question, selected_tool, tool_result, final_response):
    return {
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        "user_question": user_question,
        "selected_tool": selected_tool,
        "tool_result": tool_result,
        "final_response": final_response,
        "human_review_required": "Yes" if "human review" in str(final_response).lower() else "No"
    }

audit_record = create_audit_record(
    "Is this claim eligible?",
    "claim_validation_tool",
    claim_validation_tool(sample_claim),
    "Claim can proceed to document and risk review."
)

print(json.dumps(audit_record, indent=2))

## Solution 4: Motor Claim Over Coverage Test

In [ ]:
motor_claim_high_amount = {
    "claim_id": "MOT2002",
    "policy_number": "VEH2045",
    "customer_name": "Meena Joshi",
    "claim_type": "Motor",
    "claim_amount": 350000
}

print(json.dumps(claim_validation_tool(motor_claim_high_amount), indent=2))
print(json.dumps(ai_assisted_uipath_workflow(motor_claim_high_amount), indent=2))

## Solution 5: Final AI Agent Skeleton

In [ ]:
def final_insurance_ai_agent(question, claim):
    rag_result = {
        "answer": "RAG not executed in this skeleton. Use rag_answer() after API setup.",
        "sources": []
    }

    selected_tool = detect_tool_from_question(question)

    if selected_tool == "claim_validation_tool":
        tool_result = claim_validation_tool(claim)
    elif selected_tool == "policy_lookup_tool":
        tool_result = policy_lookup_tool(claim.get("policy_number"))
    elif selected_tool == "document_checklist_tool":
        tool_result = document_checklist_tool(claim.get("claim_type"))
    else:
        tool_result = {"message": "No tool selected"}

    uipath_result = ai_assisted_uipath_workflow(claim)

    audit = create_audit_record(
        question,
        selected_tool,
        tool_result,
        uipath_result
    )

    return {
        "question": question,
        "rag_result": rag_result,
        "selected_tool": selected_tool,
        "tool_result": tool_result,
        "uipath_result": uipath_result,
        "audit": audit
    }

final_agent_result = final_insurance_ai_agent(
    "Is this claim eligible under the policy?",
    sample_claim
)

print(json.dumps(final_agent_result, indent=2))

# 30. Review Questions

1. What problem does RAG solve in insurance policy Q&A?
2. Why is vector search useful for policy documents?
3. Why should RAG answers include source titles?
4. What is the role of MCP-style tools in enterprise AI automation?
5. Why is tool discovery important?
6. How can LangSmith help debug LangGraph and MCP workflows?
7. How can UiPath call a Python AI service?
8. What should happen when RAG confidence is low?
9. Which AI tool access risks should be governed?
10. How can an AI agent trigger UiPath-supported workflow safely?

# 31. Day 7 Summary

You completed:

- RAG concepts with insurance examples
- Embeddings and vector storage explanation
- In-memory vector retrieval
- RAG answer generation
- JSON output for UiPath
- LangGraph + RAG workflow
- Retry and error recovery
- MCP fundamentals
- MCP-style policy lookup, claim validation and document checklist tools
- Tool discovery
- AI tool selection agent
- FastAPI MCP-style tool server concept
- LangSmith tracing use case
- UiPath REST API integration pattern
- Mock UiPath robot trigger
- Security and governance considerations
- 5 practice examples
- Final AI Agent assignment with UiPath-supported automation
- Solution snippets

## Next Step

This prepares participants for the Enterprise Capstone:

```text
AI-Powered Insurance Claims Automation Platform
```